# 03 — Text Cleaning and Normalisation

This notebook starts from the manually checked binary-labelled sentence dataset and performs the cleaning steps from the original project code.

Main steps:

- lowercase
- duplicate removal
- whitespace cleanup
- encoding repair
- contraction expansion
- selected symbol handling
- number removal
- apostrophe/possessive cleanup
- bank/place masking
- slang, abbreviation and typo normalisation
- non-English/unrecognised-word inspection

> **Ordering adjustment:** contraction expansion is performed before apostrophe removal because the original notes explicitly state that contractions such as `don't` and `wasn't` should be expanded before punctuation cleanup.

In [ ]:
from pathlib import Path
import pandas as pd
import re

PROJECT_ROOT = Path("..")
INPUT_PATH = PROJECT_ROOT / "data" / "intermediate" / "05_sentence_raw.csv"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

df_text_labelled = pd.read_csv(INPUT_PATH)

print("Loaded:", INPUT_PATH)
print("Rows:", len(df_text_labelled))
display(df_text_labelled.head())

## Install text-normalisation packages

In [ ]:
%pip install -q ftfy contractions pyenchant

In [ ]:
import ftfy
import contractions

## Lowercase and remove exact duplicate sentences

In [ ]:
df_text_labelled["review_text"] = (
    df_text_labelled["review_text"]
    .astype(str)
    .str.lower()
)

duplicate_count = df_text_labelled.duplicated(
    subset="review_text"
).sum()

print("Duplicate rows found:", duplicate_count)

before = len(df_text_labelled)

df_text_labelled = (
    df_text_labelled
    .drop_duplicates(subset="review_text", keep="first")
    .reset_index(drop=True)
)

print("Rows before:", before)
print("Rows after:", len(df_text_labelled))
print("Rows removed:", before - len(df_text_labelled))

## Fix broken encoding and expand contractions

In [ ]:
df_text_labelled["review_text"] = (
    df_text_labelled["review_text"]
    .apply(ftfy.fix_text)
    .apply(contractions.fix)
)

display(df_text_labelled[["review_text"]].head())

## Normalise whitespace

In [ ]:
df_text_labelled["review_text"] = (
    df_text_labelled["review_text"]
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

## Preserve the meaning of `>` before symbol cleanup

The original project notes convert `>` to the phrase `more than` rather than deleting it.

In [ ]:
df_text_labelled["review_text"] = (
    df_text_labelled["review_text"]
    .str.replace(">", " more than ", regex=False)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

## Remove irrelevant numbers

In [ ]:
rows_with_numbers = df_text_labelled[
    df_text_labelled["review_text"].str.contains(r"\d", regex=True, na=False)
]

print("Rows containing numbers before cleaning:", len(rows_with_numbers))

df_text_labelled["review_text"] = (
    df_text_labelled["review_text"]
    .str.replace(r"\d+", "", regex=True)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

## Remove selected punctuation and symbols

In [ ]:
df_text_labelled["review_text"] = (
    df_text_labelled["review_text"]
    .str.replace("-", " ", regex=False)
    .str.replace(r"[-–—]", " ", regex=True)
    .str.replace(r"\.{2,}", " ", regex=True)
    .str.replace("…", " ", regex=False)
    .str.replace(".", " ", regex=False)
    .str.replace(r'[&!\/,“”"()#@\?:+%]', " ", regex=True)
    .str.replace("\ufe0f", " ", regex=False)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

## Remove unnecessary apostrophes while preserving apostrophes between letters

In [ ]:
df_text_labelled["review_text"] = (
    df_text_labelled["review_text"]
    .str.replace(r"(?<![a-zA-Z])'|'(?![a-zA-Z])", " ", regex=True)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

## Replace masked system name

In [ ]:
df_text_labelled["review_text"] = (
    df_text_labelled["review_text"]
    .str.replace(r"r\*w\*\*", "system", regex=True)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

## Remove possessive `'s`

In [ ]:
df_text_labelled["review_text"] = (
    df_text_labelled["review_text"]
    .str.replace(r"\b([a-zA-Z]+)'s\b", r"\1", regex=True)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

## Manual mappings: banks, abbreviations, slang, places and typos

In [ ]:
bank_terms = {
    "cimb": "bankname",
    "rhb": "bankname",
    "uob": "bankname",
    "citi": "bankname",
    "citibank": "bankname",
    "maybank": "bankname",
    "pbb": "bankname",
    "pb": "bankname",
    "hsbc": "bankname",
    "ocbc": "bankname",
    "bank islam": "bankname",
    "bank rakyat": "bankname",
    "public bank": "bankname",
    "public bank berhad": "bankname"
}

english_short_forms = {
    "u": "you",
    "k": "",
    "comm": "commission",
    "dept": "department",
    "mgmt": "management",
    "lvl": "level",
    "yr": "year",
    "yrs": "years",
    "mon": "monday",
    "fri": "friday",
    "eg": "example",
    "ie": "that is",
    "ur": "your",
    "ok": "okay",
    "tgter": "together",
    "hlp": "help",
    "eq": "emotional intelligence",
    "flexi": "flexible",
    "lol": "laughing out loud",
    "bos": "boss",
    "ocr": "optical character recognition",
    "ai": "artificial intelligence",
    "ull": "you will",
    "eventho": "even though",
    "que": "queue",
    "geas": "graduate employability enhancement scheme"
}

work_abbreviations = {
    "kpi": "key performance indicator",
    "wfh": "work from home",
    "wlb": "work life balance",
    "ot": "overtime",
    "ots": "overtime",
    "sop": "standard operating procedure",
    "sops": "standard operating procedures",
    "hrbp": "human resource business partner",
    "sr": "service request",
    "oic": "officer in charge",
    "oics": "officers in charge",
    "hq": "headquarters",
    "tl": "team leader",
    "tm": "team manager",
    "al": "annual leave",
    "hl": "housing loan",
    "hp": "car loan",
    "hra": "housing rent allowance",
    "da": "dearness allowance",
    "jd": "job description",
    "mc": "medical certificate",
    "sme": "small and medium sized enterprises",
    "iso": "international organization for standardization",
    "epf": "employee provident fund",
    "cia": "chief internal auditor",
    "concall": "conference call",
    "ees": "employee engagement surveys",
    "bau": "business as usual",
    "ceo": "chief executive officer",
    "rorg": "reorganization"
}

malay_slang = {
    "lah": "",
    "tak": "not",
    "x": "not",
    "pandai bodek": "suck up",
    "menara": "tower",
    "tai chi": "avoid responsibility",
    "buli": "bully"
}

place_terms = dict.fromkeys(
    [
        "alor setar",
        "alor",
        "setar",
        "ipoh",
        "kedah",
        "penang",
        "malaysia",
        "kinabalu"
    ],
    "placename"
)

typo_corrections = {
    "stayback": "stay back",
    "commision": "commission",
    "commisions": "commissions",
    "commisioning": "commissioning",
    "comissioning": "commissioning",
    "work‑life balance": "work life balance",
    "hierarch": "hierarchy",
    "preffered": "preferred",
    "chnage": "change",
    "alot": "a lot",
    "beed": "need",
    "imcludes": "includes",
    "diaturb": "disturbed",
    "teamates": "teammates",
    "increament": "increment",
    "profesional": "professional",
    "enda": "end",
    "implimented": "implemented",
    "enviroment": "environment",
    "reffaller": "referral",
    "collegues": "colleagues",
    "newcommer": "newcomer",
    "oppertunity": "opportunity",
    "everday": "everyday",
    "jobscope": "job scope",
    "decaded": "outdated",
    "overprocess": "overprocessed",
    "manangement": "management",
    "knowledges": "knowledge",
    "pleasent": "pleasant",
    "unproper": "improper",
    "contidence": "confidence",
    "internaltional": "international",
    "advices": "advice",
    "friendlt": "friendly",
    "gor": "for",
    "goodplace": "good place",
    "paperworks": "paperwork",
    "fundwork": "fun work",
    "experince": "experience",
    "worklife": "work life",
    "rwally": "really",
    "renumeration": "remuneration",
    "unhumanity": "inhumanity",
    "persomal": "personal"
}

word_map = {}
word_map.update(english_short_forms)
word_map.update(work_abbreviations)
word_map.update(malay_slang)
word_map.update(place_terms)
word_map.update(typo_corrections)

def manual_map_text(text):
    text = str(text).lower()

    phrase_map = {}
    phrase_map.update(bank_terms)
    phrase_map.update({
        k: v for k, v in word_map.items()
        if " " in k
    })

    for old, new in sorted(
        phrase_map.items(),
        key=lambda x: len(x[0]),
        reverse=True
    ):
        text = re.sub(
            r"\b" + re.escape(old) + r"\b",
            new,
            text
        )

    single_word_map = {
        k: v for k, v in word_map.items()
        if " " not in k
    }

    words = text.split()
    mapped_words = [
        single_word_map.get(word, word)
        for word in words
    ]

    return re.sub(
        r"\s+",
        " ",
        " ".join(mapped_words)
    ).strip()

df_text_labelled["review_text"] = (
    df_text_labelled["review_text"]
    .apply(manual_map_text)
)

## Inspect unrecognised English words

`pyenchant` may require the system Enchant library. In Google Colab run the apt command first.

In [ ]:
# For Google Colab only, uncomment if Enchant is not installed:
# !apt-get update -qq
# !apt-get install -y enchant-2

In [ ]:
try:
    import enchant

    d = enchant.Dict("en_US")

    ignore_or_keep_words = {
        "recognised", "programmes", "sunday", "wednesday", "adulting",
        "benchmarking", "siloed", "learnt", "digitalize", "hospitalisation",
        "chinese", "reportings", "townhall", "upskilling", "christmas",
        "programme", "favouritism", "organisation", "roleplay", "hahaha",
        "travelling", "monday", "friday", "glassdoor", "chatgpt",
        "datalake", "softskills", "islamic", "trainings", "desking",
        "mentorship", "flexcare", "freshie", "elearning", "employability",
        "specialised", "centric", "centre", "gaslighting", "positivity",
        "kickstart", "frontend", "placename", "bankname"
    }

    ignore_words = ignore_or_keep_words.union(
        set(typo_corrections.keys())
    )

    def find_non_english_words(text):
        words = re.findall(
            r"\b[a-zA-Z]+\b",
            str(text).lower()
        )

        return sorted({
            word
            for word in words
            if word not in ignore_words
            and not d.check(word)
        })

    df_text_labelled["non_english_words"] = (
        df_text_labelled["review_text"]
        .apply(find_non_english_words)
    )

    rows_non_english = df_text_labelled[
        df_text_labelled["non_english_words"].str.len() > 0
    ]

    display(
        rows_non_english[
            ["review_text", "non_english_words"]
        ].reset_index()
    )

except Exception as e:
    print("Enchant inspection skipped:", e)
    df_text_labelled["non_english_words"] = [[] for _ in range(len(df_text_labelled))]

## Save cleaned and normalised sentences

In [ ]:
output_path = PROCESSED_DIR / "06_sentences_processed.csv"

df_text_labelled.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig"
)

print("Saved:", output_path)
print("Rows:", len(df_text_labelled))
display(df_text_labelled.head())